# WFTFC freq-only Fine-tuning

In this notebook, we fine-tune the pre-trained base model of WFTFC in a closed world scenario with only freq feature. 

We evaluate NetCLR using two datasets: AWF and Drift datasets. 

N defines the number of labeled samples that we use for fine-tuning.  

In [1]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
from __future__ import unicode_literals

import warnings
warnings.filterwarnings('ignore')
import numpy as np

from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import RandomSampler, SequentialSampler
import torch
from torch import nn
import torch.nn.functional as F
from torch import optim
from torch.autograd import Variable
# from torchvision import datasets, transforms
import tqdm
import pickle
import argparse
from torch.cuda.amp import GradScaler, autocast

import random
import sys
import os
import collections
from sklearn.model_selection import train_test_split

## GPU Allocation

In [2]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu", 0)
kwargs = {'num_workers': 0, 'pin_memory': True} if use_cuda else {}
print (f'Device: {device}')

Device: cuda:0


## Parameters

In [3]:
batch_size = 32

## Loading the Fine-tuning Datasets

In [4]:
DATASET = 'AWF' # 'Drift'

if DATASET == 'AWF':    
    data_path = './datasets/awf2_freq.npz' # AWF-attack
    # data = pickle.load(open(f'{data_path}', 'rb'))
    data = np.load(data_path)

# awf2_freq.npz
x_total = data['x_freq']
y_total = data['y']

x_train_total, x_test_total, y_train_total, y_test_total = train_test_split(
    x_total, y_total, test_size=0.2, random_state=42, stratify=y_total)

# 如果你的目标域只有一个测试集，可以将其同时赋值给 sup 和 inf，或者只保留一个
x_test_sup = x_test_total
y_test_sup = y_test_total
# x_test_inf = x_test_total # 或者保留为空，取决于你是否还需要对比
# y_test_inf = y_test_total

num_classes = len(np.unique(y_train_total))
print ("Number of classes:", num_classes)

Number of classes: 103


In [5]:
# --- 修改后的 IN[5] ---
# 确保打印的是你实际加载进来的变量
print (f'Train shape: {x_train_total.shape}')
print (f'Test shape: {x_test_sup.shape}') # 假设你统一用了 sup 作为测试

Train shape: (206000, 2500)
Test shape: (51500, 2500)


In [6]:
# This function randomly samples N traces per website
def sample_traces(x, y, N):
    train_index = []
    
    for c in range(num_classes):
        idx = np.where(y == c)[0]
        idx = np.random.choice(idx, min(N, len(idx)), False)
        train_index.extend(idx)
        
    train_index = np.array(train_index)
    np.random.shuffle(train_index)
    
    x_train = x[train_index]
    y_train = y[train_index]
    
    return x_train, y_train

## Backbone Model

In [7]:
class FCN(nn.Module):
    def __init__(self,
                 n_channels=1,
                 out_channels=128):
        super().__init__()

        kernel_size = 8

        self.conv_block1 = nn.Sequential(
            nn.Conv1d(n_channels, 32,
                      kernel_size=kernel_size,
                      stride=1,
                      padding=kernel_size//2,
                      bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2,2,padding=1),
            nn.Dropout(0.35)
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv1d(32,64,
                      kernel_size=kernel_size,
                      stride=1,
                      padding=kernel_size//2,
                      bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2,2,padding=1)
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv1d(64,out_channels,
                      kernel_size=kernel_size,
                      stride=1,
                      padding=kernel_size//2,
                      bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.MaxPool1d(2,2,padding=1)
        )

        # 不固定长度
        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.feature_dim = out_channels

    def forward(self,x):

        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)

        x = self.global_pool(x)

        x = x.squeeze(-1)

        return x



In [8]:
class FCNSimCLR(nn.Module):

    def __init__(self, backbone, out_dim=128):
        super().__init__()

        self.backbone = backbone

        self.projector = nn.Sequential(
            nn.Linear(backbone.feature_dim, backbone.feature_dim),
            nn.BatchNorm1d(backbone.feature_dim),
            nn.ReLU(inplace=True),
            nn.Linear(backbone.feature_dim, out_dim)
        )

    def forward(self, x):
        h = self.backbone(x)      # (B,256)
        z = self.projector(h)     # (B,128)
        return z

## Data Loader

In [9]:
class Data(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
        
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    
    def __len__(self):
        return len(self.x)

## Loading the Pre-trained Model

In [22]:
def load_checkpoint_for_fcn():
    # 1. 实例化你的 FCN 模型
    # 关键：确保这里的 out_channels=256，和预训练时 backbone 的结构完全一致
    model = FCN(n_channels=1, out_channels=256).to(device)

    # 2. 加载预训练的 FCNSimCLR 模型权重
    checkpoint = torch.load('./checkpoints/WFTFC/WFTFC_freq_FCN_epoch_100.pth.tar', map_location=device)

    # 3. 创建一个新的字典，只存放 backbone 的权重，并去掉 'backbone.' 前缀
    new_state_dict = {}
    for k, v in checkpoint.items():
        if k.startswith('backbone.'):
            name = k[len("backbone."):]  # 去掉 'backbone.' 前缀
            new_state_dict[name] = v

    # 4. 加载权重
    # 因为模型结构完全匹配，这里应该使用 strict=True
    # 如果有不匹配的层，会直接报错，这样你才能发现问题
    model.load_state_dict(new_state_dict, strict=True)
    
    print("✅ Backbone 权重加载成功！")
    return model

## Initating Test Data Loaders

In [16]:
## 用的AWF2只有一个测试集无优劣之分
# test_dataset_inf = Data(x_test_inf, y_test_inf)
# test_loader_inf = DataLoader(test_dataset_inf, batch_size=batch_size, drop_last=True)

test_dataset_sup = Data(x_test_sup, y_test_sup)
test_loader_sup = DataLoader(test_dataset_sup, batch_size=batch_size, drop_last=True)

## Function for Train and Test

In [17]:
def train(model, device, train_loader, optimizer):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data = data.view(data.size(0), 1, data.size(1)).float().to(device)
        target = target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        # print (output.size())
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx%100 == 0:
            print ("Loss: {:0.6f}".format(loss.item()))
    
def test(model, device, loader):
    model.eval()
    correct = 0
    temp = 0
    with torch.no_grad():
        for data, target in loader:
            data = data.view(data.size(0), 1, data.size(1)).float().to(device)
            target = target.to(device)
            
            output = model(data)
            output = torch.softmax(output, dim=1)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).float().sum().item()
    return correct / len(loader.dataset)

## Running for 5 Times

In [18]:
# N defines the number of labeled samples we use to perform fine-tuning
N = 5

In [24]:
# accuracies_inf = []
accuracies_sup = []
for _ in range(5): # 测试5次，后面取平均值
    x_train, y_train = sample_traces(x_train_total, y_train_total, N)
    
    print ("Input size:", x_train.shape, y_train.shape)
    
    train_dataset = Data(x_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    
    model = load_checkpoint_for_fcn()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    
    
    best_acc_inf = 0
    best_acc_sup = 0
    for epoch in range(101):
        print ('Epoch: ', epoch)
        train(model, device, train_loader, optimizer)
        
        # acc_inf = test(model, device, test_loader_inf)
        acc_sup = test(model, device, test_loader_sup)
        
        # best_acc_inf = max(best_acc_inf, acc_inf)
        best_acc_sup = max(best_acc_sup, acc_sup)
        
        if epoch%10 == 0:
            # print (f"Accuracy on inferior dataset: {acc_inf*100:.2f}")
            print (f"Accuracy on superior dataset: {acc_sup*100:.2f}")
                
    # accuracies_inf.append(best_acc_inf)
    accuracies_sup.append(best_acc_sup)
    
    
    print ('------------------------------------------------')

Input size: (515, 2500) (515,)
✅ Backbone 权重加载成功！
Epoch:  0
Loss: 5.543621
Accuracy on superior dataset: 0.04
Epoch:  1
Loss: 5.584215
Epoch:  2
Loss: 5.521927
Epoch:  3
Loss: 5.494890
Epoch:  4
Loss: 5.576125
Epoch:  5
Loss: 5.547039
Epoch:  6
Loss: 5.479731
Epoch:  7
Loss: 5.523483
Epoch:  8
Loss: 5.475071
Epoch:  9
Loss: 5.473415
Epoch:  10
Loss: 5.438983
Accuracy on superior dataset: 0.17
Epoch:  11
Loss: 5.488770
Epoch:  12
Loss: 5.450424
Epoch:  13
Loss: 5.477304
Epoch:  14
Loss: 5.496918
Epoch:  15
Loss: 5.442794
Epoch:  16
Loss: 5.484514
Epoch:  17
Loss: 5.433875
Epoch:  18
Loss: 5.416954
Epoch:  19
Loss: 5.438246
Epoch:  20
Loss: 5.472033
Accuracy on superior dataset: 1.01
Epoch:  21
Loss: 5.409754
Epoch:  22
Loss: 5.396148
Epoch:  23
Loss: 5.426003
Epoch:  24
Loss: 5.373385
Epoch:  25
Loss: 5.282013
Epoch:  26
Loss: 5.404880
Epoch:  27
Loss: 5.419000
Epoch:  28
Loss: 5.403209
Epoch:  29
Loss: 5.404304
Epoch:  30
Loss: 5.290200
Accuracy on superior dataset: 2.04
Epoch:  31
Los

In [25]:
# accuracies_inf = np.array(accuracies_inf)
accuracies_sup = np.array(accuracies_sup)

# print (f"Test accuracy on inferior traces: avg -> {np.mean(accuracies_inf)*100:.1f}, std -> {np.std(accuracies_inf)*100:.1f}")
print (f"Test accuracy on Superior traces: avg -> {np.mean(accuracies_sup)*100:.1f}, std -> {np.std(accuracies_sup)*100:.1f}")

Test accuracy on Superior traces: avg -> 4.6, std -> 0.3
